Task 11

In [1]:
import numpy
import gensim
import csv
from gensim.models import KeyedVectors

In [2]:
!wget http://dl.turkunlp.org/TKO_7095_2023/12.zip
!wget http://dl.turkunlp.org/TKO_7095_2023/42.zip



7[Files: 0  Bytes: 0  [0 B/s] Re]87[http://dl.turkunlp.org/TKO_709]87Saving '12.zip'
87[Files: 0  Bytes: 0  [0 B/s] Re]8712.zip                 0% [>                             ]    5.45M    --.-KB/s87[Files: 0  Bytes: 0  [0 B/s] Re]8712.zip                 4% [>                             ]   26.27M   20.80MB/s87[Files: 0  Bytes: 0  [0 B/s] Re]8712.zip                 9% [=>                            ]   55.40M   24.96MB/s87[Files: 0  Bytes: 0  [0 B/s] Re]8712.zip                13% [===>                          ]   81.64M   25.38MB/s87[Files: 0  Bytes: 0  [0 B/s] Re]8712.zip                18% [====>                         ]  106.38M   25.22MB/s87[Files: 0  Bytes: 0  [0 B/s] Re]8712.zip                19% [====>                         ]  116.82M   22.26MB/s87[Files: 0  Bytes: 0  [0 B/s] Re]8712.zip                22% [=====>                        ]  132.07M   21.09MB/s87[Files: 0  Bytes: 0  [0 B/s] Re]8712.zip                27% [=======>

In [3]:
!unzip -o 12.zip
!mv model.bin en.bin
!unzip -o 42.zip
!mv model.bin fi.bin

Archive:  12.zip
  inflating: meta.json               
  inflating: model.bin               
  inflating: model.txt               
  inflating: README                  
Archive:  42.zip
  inflating: LIST                    
  inflating: meta.json               
  inflating: model.bin               
  inflating: model.txt               
  inflating: README                  


In [4]:
wv_emb_en=KeyedVectors.load_word2vec_format("en.bin", limit=100000, binary=True)
wv_emb_fi=KeyedVectors.load_word2vec_format("fi.bin", limit=100000, binary=True)

Nearest Wird Lookup

In [5]:
result = wv_emb_en.similar_by_word('cat')
for most_similar_key, similarity in result[:10]:
    print(f"{most_similar_key}: {similarity:.4f}")

dog: 0.8098
cats: 0.7977
mutt: 0.7505
kitten: 0.7471
feline: 0.7438
puppy: 0.7252
raccoon: 0.7183
pooch: 0.7116
squirrel: 0.6906
kittens: 0.6856


Word analogy

In [6]:
# B     A      C
# Paris-France+Sweden= ___?
#
# i.e. France is to Paris as Sweden is to X
wv_emb_en.most_similar(positive=["Paris","Sweden"],negative=["France"])

[('Stockholm', 0.7338932752609253),
 ('Malmo', 0.5458161234855652),
 ('Helsinki', 0.5444939732551575),
 ('Goteborg', 0.5421050190925598),
 ('Swedish', 0.5309098362922668),
 ('Malmoe', 0.5198634266853333),
 ('Oslo', 0.5004472732543945),
 ('Gothenburg', 0.4957912266254425),
 ('STOCKHOLM', 0.48791584372520447),
 ('Copenhagen', 0.47769418358802795)]

Bilingual dictionaries

In [7]:
# Grab the data
!wget https://raw.githubusercontent.com/codogogo/xling-eval/master/bli_datasets/en-fi/yacle.test.freq.2k.en-fi.tsv
!wget https://raw.githubusercontent.com/codogogo/xling-eval/master/bli_datasets/en-fi/yacle.train.freq.5k.en-fi.tsv



7[Files: 0  Bytes: 0  [0 B/s] Re]87[https://raw.githubusercontent.]87Saving 'yacle.test.freq.2k.en-fi.tsv'
87yacle.test.freq.2k.e 100% [=============================>]   17.15K    --.-KB/s87HTTP response 200  [https://raw.githubusercontent.com/codogogo/xling-eval/master/bli_datasets/en-fi/yacle.test.freq.2k.en-fi.tsv]
87yacle.test.freq.2k.e 100% [=============================>]   17.15K    --.-KB/s87[Files: 1  Bytes: 17.15K [40.09]8

7[Files: 0  Bytes: 0  [0 B/s] Re]87[https://raw.githubusercontent.]87Saving 'yacle.train.freq.5k.en-fi.tsv'
87yacle.train.freq.5k. 100% [=============================>]   37.21K    --.-KB/s87HTTP response 200  [https://raw.githubusercontent.com/codogogo/xling-eval/master/bli_datasets/en-fi/yacle.train.freq.5k.en-fi.tsv]
87yacle.train.freq.5k. 100% [=============================>]   37.21K    --.-KB/s87[Files: 1  Bytes: 37.21K [77.68]8

In [8]:
!cat yacle.test.freq.2k.en-fi.tsv | head -n 10

dedication	omistautuminen
desires	toiveet
dismissed	hylätty
psychic	psyykkinen
cracks	halkeamia
establishments	laitokset
efficacy	tehokkuus
prestige	arvovalta
cocaine	kokaiini
accelerated	kiihtyi


In [9]:
pairs_train=[] #These will be pairs of (source,target) i.e. (Finnish, English) words used to induce the matrix M
pairs_test=[]  #same but for testing, so we should make sure there is absolutely no overlap between the train and test data
               #let's do it so that not one word in the test is is seen in any capacity in the training data

def get_vectors(fname):
    """
    Read the pairs from the file `fname`
    """
    pairs=[]
    with open(fname) as f:
        r = csv.reader(f,delimiter="\t") #the file is a .tsv i.e. tab-separated-values
        for en_word,fi_word in r:
            #I will reverse the order here, go from Finnish as the source, to English as the target
            #That way it will be easier to check how this works using English as the target, which we all understand
            pairs.append((fi_word,en_word))
        return pairs

train_data=get_vectors("yacle.train.freq.5k.en-fi.tsv")
test_data=get_vectors("yacle.test.freq.2k.en-fi.tsv")
print(train_data[:10])
print(len(train_data))
print(test_data[:10])
print(len(test_data))



[('of', 'of'), ('että', 'to'), ('sisään', 'in'), ('varten', 'for'), ('on', 'is'), ('päällä', 'on'), ('että', 'that'), ('mennessä', 'by'), ('Tämä', 'this'), ('kanssa', 'with')]
5000
[('omistautuminen', 'dedication'), ('toiveet', 'desires'), ('hylätty', 'dismissed'), ('psyykkinen', 'psychic'), ('halkeamia', 'cracks'), ('laitokset', 'establishments'), ('tehokkuus', 'efficacy'), ('arvovalta', 'prestige'), ('kokaiini', 'cocaine'), ('kiihtyi', 'accelerated')]
2000


Get the embeddings

In [10]:
def build_arrays(pairs,emb1,emb2,avoid=set()):
    """
    `pairs`: pairs of (fi,en) words
    `emb1`: source side (here Finnish) embeddings
    `emb2`: target side (here English) embeddings
    `avoid`: a set of words to avoid/ignore (will be used when building test data, to avoid train data)
    """
    vecs1,vecs2,filtered_pairs=[],[],[]  #vectors for source words, vectors for target words, and the word pairs themselves, i.e. three same-length lists
    for w1,w2 in pairs: #Go over all pairs that we got
        # check if both vectors are available, and none of the words is to be avoided
        if w1 in emb1 and w2 in emb2 and w1 not in avoid and w2 not in avoid:
            #passed!
            vecs1.append(emb1[w1]) #source-side embedding, the KeyedVectors object can be queried as if it was a dictionary, returns the embedding as 1-dim array
            vecs2.append(emb2[w2]) #target-side embeddings
            filtered_pairs.append((w1,w2)) #remember the pair
    #Now we vstack() which turns the lists of embeddings into 2-dim array
    return numpy.vstack(vecs1),numpy.vstack(vecs2),filtered_pairs

# Gather the train data first
array_train_fi,array_train_en,pairs_train=build_arrays(train_data,wv_emb_fi,wv_emb_en)
# Now build the set of all words seen in training, so we can avoid them when building the test set. Note that "|" is set union operator
everything_in_train=set(s for s,t in pairs_train)|set(t for s,t in pairs_train)
# Test data next, avoiding the words from the training data:
array_test_fi,array_test_en,pairs_test=build_arrays(test_data,wv_emb_fi,wv_emb_en,avoid=everything_in_train)

In [11]:
# Let's be super-sure there absolutely is no overlap of any kind!
print("Overlap between train pairs and test pairs:",len(set(pairs_train) & set(pairs_test))) # & is set intersection operator, intersection between train and test should be empty
src_train=set(src_w for src_w,tgt_w in pairs_train) #train source words
tgt_train=set(tgt_w for src_w,tgt_w in pairs_train) #train target words
src_test=set(src_w for src_w,tgt_w in pairs_test)   #test source words
tgt_test=set(tgt_w for src_w,tgt_w in pairs_test)   #test target words
print("Overlap between train fi words and test fi words:",len(src_train & src_test))
print("Overlap between train en words and test en words:",len(tgt_train & tgt_test))

Overlap between train pairs and test pairs: 0
Overlap between train fi words and test fi words: 0
Overlap between train en words and test en words: 0


Mapping matrix

In [12]:
# This code was written by GPT4, but in a bit of a twisted form, so I modified it
# to better correspond to the formulae in the lecture

def learn_transformation_matrix(source, target):
    # Compute the pseudo-inverse of the source matrix
    source_pseudo_inverse = numpy.linalg.pinv(source) # This implements (S^T S)^-1 S^T  needed in the least-squares formula in the lecture slides
    # Compute the transformation matrix M using least squares method
    M = numpy.matmul(source_pseudo_inverse,target)  #...and this multiplies by T from right completing the formula in the slides ... two lines(!)
    return M

# fi -> en matrix
M=learn_transformation_matrix(array_train_fi,array_train_en)

# Ha ha well that was easy

In [13]:
print("Source (fi) shape",array_train_fi.shape)
print("Target (en) shape",array_train_en.shape)
print("M shape",M.shape)

Source (fi) shape (4506, 100)
Target (en) shape (4506, 300)
M shape (100, 300)


In [14]:
# And now we transform the source (Finnish) test embeddings into the English embedding space
# using the matrix M
test_fi_transformed=numpy.matmul(array_test_fi,M) # This corresponds to SM in the lecture slides, i.e. source transformed by M to the target embedding space
print("Transformed shape:",test_fi_transformed.shape)
numpy.square(numpy.subtract(test_fi_transformed, array_test_en)).mean() #This is the mean square error of the actual target, and the transformed source, looks small enough :)

Transformed shape: (1285, 300)


np.float32(0.0023262969)

In [15]:
for i,(w1,w2) in enumerate(pairs_test[:50]):
    print(f"{w1} (in English {w2}):")
    nearest_neighbors=wv_emb_en.similar_by_vector(test_fi_transformed[i]) #lookup the nearest
    #nearest_neigbors will be tuples (word,similarity_value)
    eng_words=[w for w, score in nearest_neighbors] #just grab the words
    print(f"   ",", ".join(eng_words)) #...and print then ,-separated
    print()

# It cannot be stressed enough, that none of the words in the test data were seen
# during the induction of the transformation matrix M
#
# We can observe some direct top-1 hits, and in general we see
# the mapping maps the vector very close to the correct place
# in my view, this is quite impressive :)

toiveet (in English desires):
    desires, importantly, Certainly, qualities, ideas, perspectives, desire, indeed, sense, notions

psyykkinen (in English psychic):
    cognitive, physiological, behavioral, physical, neurological, mental, disorders, empathy, therapy, interpersonal

halkeamia (in English cracks):
    crevices, vegetation, gullies, surfaces, ridges, walls, limestone, reddish, mottled, sediment

kokaiini (in English cocaine):
    additives, pesticides, substances, caffeine, foods, carcinogenic, medications, drugs, side-effects, chemicals

kiihtyi (in English accelerated):
    slowed, worsened, accelerated, surged, exacerbated, spurred, stagnated, slackened, fueled, ebbed

huippu (in English pinnacle):
    magnificent, breathtaking, ideal, marvelous, majestic, perfect, beautiful, gorgeous, fabulous, awesome

edellä (in English supra):
    therefore, although, instances, indeed, simply, Furthermore, merely, Consequently, fact, actually

päärynä (in English pear):
    melon, 